In [ ]:
! pip install scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 scipy==1.16.1

### Randomized Search

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import scipy.stats as st

# 1) Load data
X, y = load_breast_cancer(return_X_y=True)

# 2) Hold out a test set for final, unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Define model
base_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# 4) Randomized search space (distributions + sensible ranges)
param_distributions = {
    "n_estimators": st.randint(100, 600),   # 100–599 trees
    "max_depth": st.randint(3, 21),         # 3–20
    "max_features": st.uniform(0.3, 0.7),   # 0.3–1.0 fraction of features
    "min_samples_split": st.randint(2, 11), # 2–10
    "min_samples_leaf": st.randint(1, 5),   # 1–4
    "bootstrap": [True, False],
    "class_weight": [None, "balanced"],
}

# 5) Cross-validation config
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6) Randomized Search (fewer trials, faster than full grid)
rnd_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=0,
    refit=True,               # refit on full training set using the best params
    return_train_score=True
)

# 7) Fit randomized search on the TRAINING split only
rnd_search.fit(X_train, y_train)

print("Best hyperparameters (RandomizedSearchCV):", rnd_search.best_params_)
print("Best mean CV ROC AUC: {:.4f}".format(rnd_search.best_score_))

best_model = rnd_search.best_estimator_

# 8) Tune the decision threshold using OOF predictions on TRAINING data
def optimal_threshold_by_youden(y_true, y_score):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y_true, y_score)
    return thr[np.argmax(tpr - fpr)]

oof_proba = cross_val_predict(
    best_model, X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

opt_thresh = optimal_threshold_by_youden(y_train, oof_proba)
print("Chosen decision threshold (Youden's J on CV predictions): {:.3f}".format(opt_thresh))

# 9) Final evaluation on the HELD-OUT TEST set (used only once)
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= opt_thresh).astype(int)

test_auc = roc_auc_score(y_test, test_proba)
test_acc = accuracy_score(y_test, test_pred)

print("\n=== Test Performance on Held-Out Set ===")
print("Test ROC AUC: {:.4f}".format(test_auc))
print("Test Accuracy (with tuned threshold): {:.4f}".format(test_acc))
print("\nClassification Report:\n", classification_report(y_test, test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, test_pred))

### Grid Search

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# 1) Load data
X, y = load_breast_cancer(return_X_y=True)

# 2) Hold out a test set for final, unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Define model and hyperparameter grid
model = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

# 4) Grid search with 5-fold CV on training data
# Use ROC AUC; refit the best by AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    refit=True
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best mean CV ROC AUC: {:.4f}".format(search.best_score_))

# 5) Evaluate once on the held-out test set
best_model = search.best_estimator_
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)  # threshold at 0.5 (simple; no tuning)

test_auc = roc_auc_score(y_test, y_prob)
test_acc = accuracy_score(y_test, y_pred)

print("\n=== Test Performance on Held-Out Set ===")
print("Test ROC AUC: {:.4f}".format(test_auc))
print("Test Accuracy: {:.4f}".format(test_acc))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))